In [3]:
# --- Cell 1: Unzip the data and load it into training arrays ---

import zipfile
import os
import numpy as np

# Unzip MP_Data.zip into Colab's working directory.
with zipfile.ZipFile("MP_Data.zip", "r") as zip_ref:
    zip_ref.extractall(".")

# This should match the 5 word-folders you created on your Windows machine.
# We sort them so the word-to-number mapping is always consistent, no matter
# what order the operating system lists folders in.
DATA_PATH = "MP_Data"
actions = sorted(os.listdir(DATA_PATH))
print("Found word folders:", actions)

# label_map turns each word into a number, e.g. {"hello": 0, "no": 1, ...}.
# Neural networks work with numbers, not text, so this is how we'll tell the
# model which word each example represents.
label_map = {label: num for num, label in enumerate(actions)}
print("Label map:", label_map)

sequences = []  # will hold all our (30, 63) landmark arrays
labels = []     # will hold the matching word-number for each array

for action in actions:
    action_folder = os.path.join(DATA_PATH, action)
    for filename in os.listdir(action_folder):
        file_path = os.path.join(action_folder, filename)
        sequence = np.load(file_path)  # shape: (30, 63)
        sequences.append(sequence)
        labels.append(label_map[action])

# Convert our Python lists into NumPy arrays, which TensorFlow expects.
# X shape: (total_examples, 30, 63) - our input data
# y shape: (total_examples,) - the correct word-number for each example
X = np.array(sequences)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

Found word folders: ['hello', 'no', 'please', 'thank_you', 'yes']
Label map: {'hello': 0, 'no': 1, 'please': 2, 'thank_you': 3, 'yes': 4}
X shape: (41, 30, 63)
y shape: (41,)


In [4]:
# --- New Cell: Augment the dataset (insert this AFTER Cell 1, BEFORE Cell 2) ---

import numpy as np

def jitter(sequence, noise_level=0.01):
    """Adds small random noise to every coordinate in the sequence."""
    noise = np.random.normal(loc=0.0, scale=noise_level, size=sequence.shape)
    return sequence + noise

def time_warp(sequence, speed_factor):
    """
    Stretches or compresses the sequence in time, then resizes it back to
    30 frames so it stays compatible with our model's fixed input shape.
    speed_factor < 1.0 = slower (stretched out), > 1.0 = faster (compressed).
    """
    original_length = sequence.shape[0]
    # Pick new frame indices, spaced according to the speed factor, then
    # squeeze/stretch them back to exactly `original_length` points.
    warped_indices = np.linspace(0, original_length - 1, int(original_length / speed_factor))
    warped_indices = np.clip(warped_indices, 0, original_length - 1)
    # Resample back down/up to exactly the original 30 frames.
    final_indices = np.linspace(0, len(warped_indices) - 1, original_length).astype(int)
    resampled = sequence[warped_indices.astype(int)][final_indices]
    return resampled

# For every real example, create a few augmented variations of it.
augmented_sequences = []
augmented_labels = []

AUGMENTATIONS_PER_EXAMPLE = 4  # how many extra copies to make of each example

for i in range(len(X)):
    original_seq = X[i]
    original_label = y[i]

    for _ in range(AUGMENTATIONS_PER_EXAMPLE):
        seq = original_seq.copy()

        # Randomly apply jitter.
        seq = jitter(seq, noise_level=0.01)

        # Randomly apply a mild speed variation (between 0.85x and 1.15x).
        speed = np.random.uniform(0.85, 1.15)
        seq = time_warp(seq, speed)

        augmented_sequences.append(seq)
        augmented_labels.append(original_label)

augmented_sequences = np.array(augmented_sequences)
augmented_labels = np.array(augmented_labels)

# Combine the original real data with the new augmented data.
X_expanded = np.concatenate([X, augmented_sequences], axis=0)
y_expanded = np.concatenate([y, augmented_labels], axis=0)

print("Original X shape:", X.shape)
print("Expanded X shape:", X_expanded.shape)
print("Expanded y shape:", y_expanded.shape)

Original X shape: (41, 30, 63)
Expanded X shape: (205, 30, 63)
Expanded y shape: (205,)


In [5]:
# --- Cell 2: Prepare data for training ---

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Turn y from plain numbers (0,1,2,3,4) into one-hot vectors.
# e.g. label 0 ("hello") becomes [1,0,0,0,0]
#      label 4 ("yes")   becomes [0,0,0,0,1]
y_categorical = to_categorical(y_expanded).astype(int)

# Split into training data (what the model learns from) and test data
# (held back, used only to check performance afterward).
# test_size=0.15 keeps the test set small since we don't have much data
# to spare - with more data later, 0.2-0.3 would be more typical.
X_train, X_test, y_train, y_test = train_test_split(
    X_expanded, y_categorical, test_size=0.15, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (174, 30, 63)
X_test shape: (31, 30, 63)
y_train shape: (174, 5)
y_test shape: (31, 5)


In [6]:
# --- Cell 3: Build the LSTM model ---

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

num_classes = len(actions)  # should be 5

model = Sequential([
    # Input layer: tells the model to expect sequences of 30 frames,
    # each frame being 63 numbers (our landmark coordinates).
    Input(shape=(30, 63)),

    # First LSTM layer, 32 units. "return_sequences=True" means it passes
    # its full frame-by-frame output to the next layer, not just a final
    # summary - needed because we're stacking another LSTM layer after it.
    LSTM(32, return_sequences=True, activation='relu'),

    # Second LSTM layer, 64 units. "return_sequences=False" (the default)
    # means this one DOES condense everything down to a single summary
    # vector, since no more LSTM layers follow it.
    LSTM(64, return_sequences=False, activation='relu'),

    # A regular Dense layer to further process that summary.
    Dense(32, activation='relu'),

    # Output layer: one unit per word, using "softmax" so the 5 outputs
    # become probabilities that add up to 100%.
    Dense(num_classes, activation='softmax')
])

# Compile the model: this configures HOW it learns.
# - optimizer='adam': the algorithm that adjusts the model's internal
#   numbers to reduce mistakes, step by step.
# - loss='categorical_crossentropy': the standard way to measure "how wrong"
#   a prediction was, for problems with multiple categories like this one.
# - metrics=['accuracy']: just for us to watch training progress in
#   human-readable terms (% correct), doesn't affect learning itself.
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 32)         │        12,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,365 (153.77 KB)

 Trainable params: 39,365 (153.77 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# --- Cell 4: Train the model ---

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=8,
    validation_data=(X_test, y_test)
)

Epoch 1/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.2126 - loss: 1.5556 - val_accuracy: 0.4194 - val_loss: 1.4069
Epoch 2/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.2989 - loss: 1.7160 - val_accuracy: 0.3871 - val_loss: 1.5000
Epoch 3/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.2874 - loss: 1.5124 - val_accuracy: 0.3226 - val_loss: 1.4958
Epoch 4/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.4310 - loss: 1.4675 - val_accuracy: 0.4194 - val_loss: 1.4126
Epoch 5/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5000 - loss: 1.3298 - val_accuracy: 0.4194 - val_loss: 1.1698
Epoch 6/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.4368 - loss: 1.2880 - val_accuracy: 0.5161 - val_loss: 1.1295
Epoch 7/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.3678 - loss: 1.2800 - val_accuracy: 0.5161 - val_loss: 1.1257
Epoch 8/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.4138 - loss: 1.2765 - val_accuracy: 0.

In [8]:
# --- New Cell: Save the trained model ---

model.save("asl_model.h5")
print("Model saved as asl_model.h5")

# This lets you download the file directly from Colab to your computer.
from google.colab import files
files.download("asl_model.h5")

Model saved as asl_model.h5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>